In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

In [ ]:
# Review 3 missing statistical checks and figures
import os, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Recreate the cleaned one-minute time series if this runtime is new
if 'df' not in globals() or 'Global_active_power' not in df.columns:
    archive = 'household_power_consumption.zip'
    if not os.path.exists(archive):
        urllib.request.urlretrieve('https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip', archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall('dataset_review3')
    path = [os.path.join('dataset_review3', f) for f in os.listdir('dataset_review3') if f.endswith('.txt')][0]
    df = pd.read_csv(path, sep=';', na_values='?', low_memory=False)
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
    df = df.set_index('Datetime').drop(columns=['Date', 'Time'])
    df = df.apply(pd.to_numeric, errors='coerce').interpolate(method='time').ffill().bfill()

feature = 'Global_active_power'
x = df[feature].dropna()
sample = x.sample(n=min(5000, len(x)), random_state=42)
shapiro_stat, shapiro_p = stats.shapiro(sample)
dago_stat, dago_p = stats.normaltest(sample)
skewness = stats.skew(sample)
kurtosis = stats.kurtosis(sample, fisher=True)
q1, q3 = x.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
outliers = int(((x < lower) | (x > upper)).sum())

print('REVIEW 3: NORMALITY AND OUTLIER RESULTS')
print(f'Feature: {feature}; random sample n={len(sample):,}')
print(f'Shapiro-Wilk: statistic={shapiro_stat:.6f}, p-value={shapiro_p:.6e}')
print(f"D'Agostino K2: statistic={dago_stat:.6f}, p-value={dago_p:.6e}")
print(f'Skewness={skewness:.6f}; excess kurtosis={kurtosis:.6f}')
print(f'IQR: Q1={q1:.6f}, Q3={q3:.6f}, lower={lower:.6f}, upper={upper:.6f}')
print(f'IQR outliers={outliers:,} of {len(x):,} ({outliers/len(x)*100:.2f}%)')

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(sample, bins=50, kde=True, ax=ax[0], color='#2E86AB')
ax[0].set_title('Normality Check: Global Active Power')
ax[0].set_xlabel('Global active power (kW)')
stats.probplot(sample, dist='norm', plot=ax[1])
ax[1].set_title('Normal Q-Q Plot')
fig.tight_layout(); fig.savefig('normality.png', dpi=200, bbox_inches='tight'); plt.show()

fig, ax = plt.subplots(figsize=(11, 3.5))
sns.boxplot(x=sample, ax=ax, color='#F6AE2D')
ax.axvline(lower, color='crimson', linestyle='--', label='IQR lower bound')
ax.axvline(upper, color='crimson', linestyle='--', label='IQR upper bound')
ax.set_title('IQR Outlier Check: Global Active Power')
ax.set_xlabel('Global active power (kW)')
ax.legend(); fig.tight_layout(); fig.savefig('outliers_box.png', dpi=200, bbox_inches='tight'); plt.show()

values = sample.to_numpy().reshape(-1, 1)
std_values = StandardScaler().fit_transform(values).ravel()
minmax_values = MinMaxScaler().fit_transform(values).ravel()
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, data, title in zip(ax, [sample, std_values, minmax_values], ['Original', 'Standardised (z-score)', 'Min-Max normalised']):
    sns.histplot(data, bins=45, ax=a, color='#2E86AB')
    a.set_title(title)
fig.suptitle('Scaling Comparison: Global Active Power', y=1.03)
fig.tight_layout(); fig.savefig('scaling_compare.png', dpi=200, bbox_inches='tight'); plt.show()


In [ ]:
from google.colab import files
files.download('normality.png')

In [ ]:
!wget -O household_power_consumption.txt \
"https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"

In [ ]:
import zipfile

with zipfile.ZipFile("household_power_consumption.txt") as z:
    z.extractall("dataset")

print("Dataset extracted successfully!")

In [ ]:
file_path = "dataset/household_power_consumption.txt"

df = pd.read_csv(
    file_path,
    sep=";",
    na_values="?",
    low_memory=False
)

print(df.head())

In [ ]:
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Combine Date and Time
df["Datetime"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    dayfirst=True
)

# Set Datetime as index
df.set_index("Datetime", inplace=True)

# Drop original Date and Time columns
df.drop(columns=["Date", "Time"], inplace=True)

print(df.head())

In [ ]:
# Convert all columns to numeric
df = df.apply(pd.to_numeric, errors="coerce")

print(df.dtypes)

In [ ]:
# Fill missing values using interpolation
df = df.interpolate(method="time")

# Handle any remaining missing values
df = df.ffill().bfill()

print("Missing values after cleaning:")
print(df.isnull().sum())

In [ ]:
print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

In [ ]:
import matplotlib.pyplot as plt

# Calculate daily average active power
daily_power = df["Global_active_power"].resample("D").mean()

# Plot daily consumption
plt.figure(figsize=(15, 5))

plt.plot(daily_power)

plt.title("Daily Electricity Consumption")
plt.xlabel("Date")
plt.ylabel("Global Active Power (kW)")

plt.grid(True)
plt.show()

In [ ]:
# Calculate average power for each hour
hourly_power = df["Global_active_power"].groupby(
    df.index.hour
).mean()

# Plot hourly consumption
plt.figure(figsize=(12, 5))

plt.bar(hourly_power.index, hourly_power.values)

plt.title("Average Electricity Consumption by Hour")
plt.xlabel("Hour of the Day")
plt.ylabel("Average Global Active Power (kW)")

plt.xticks(range(24))
plt.grid(axis="y")

plt.show()

In [ ]:
# Calculate monthly average active power
monthly_power = df["Global_active_power"].resample("ME").mean()

# Plot monthly consumption
plt.figure(figsize=(15, 5))

plt.plot(monthly_power.index, monthly_power.values)

plt.title("Monthly Electricity Consumption")
plt.xlabel("Month")
plt.ylabel("Average Global Active Power (kW)")

plt.grid(True)
plt.show()

In [ ]:
# Calculate weekly average active power
weekly_power = df["Global_active_power"].resample("W").mean()

# Plot weekly consumption
plt.figure(figsize=(15, 5))

plt.plot(weekly_power.index, weekly_power.values)

plt.title("Weekly Electricity Consumption")
plt.xlabel("Week")
plt.ylabel("Average Global Active Power (kW)")

plt.grid(True)
plt.show()

In [ ]:
# Calculate correlation matrix
correlation = df.corr()

# Plot heatmap
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Electricity Consumption Correlation Heatmap")
plt.show()

In [ ]:
# Top 5 peak consumption hours
peak_hours = hourly_power.sort_values(ascending=False)

print("Top 5 Peak Consumption Hours:")
print(peak_hours.head(5))

In [ ]:
# Create daily average electricity consumption
daily_data = df["Global_active_power"].resample("D").mean()

# Convert to DataFrame
daily_df = daily_data.to_frame(name="Power")

print(daily_df.head())
print("Shape:", daily_df.shape)

In [ ]:
# Create lag features
daily_df["Lag_1"] = daily_df["Power"].shift(1)
daily_df["Lag_7"] = daily_df["Power"].shift(7)
daily_df["Lag_14"] = daily_df["Power"].shift(14)

# Rolling average feature
daily_df["Rolling_7"] = daily_df["Power"].shift(1).rolling(7).mean()

# Calendar features
daily_df["Day"] = daily_df.index.day
daily_df["Month"] = daily_df.index.month
daily_df["DayOfWeek"] = daily_df.index.dayofweek

# Remove rows with missing feature values
daily_df.dropna(inplace=True)

print(daily_df.head())
print("Shape:", daily_df.shape)

In [ ]:
from sklearn.model_selection import train_test_split

# Define features and target
X = daily_df.drop(columns=["Power"])
y = daily_df["Power"]

# Chronological split
split_index = int(len(daily_df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train model
model.fit(X_train, y_train)

print("Model trained successfully!")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Create model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Train model
model.fit(X_train, y_train)

print("Model trained successfully!")

In [ ]:
y_pred = model.predict(X_test)

print("Predictions generated successfully!")
print(y_pred[:10])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Calculate metrics
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print("Model Evaluation")
print("----------------")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    y_test.index,
    y_test.values,
    label="Actual"
)

plt.plot(
    y_test.index,
    y_pred,
    label="Predicted"
)

plt.title("Actual vs Predicted Electricity Consumption")
plt.xlabel("Date")
plt.ylabel("Average Global Active Power (kW)")

plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression

# Create Linear Regression model
lr_model = LinearRegression()

# Train model
lr_model.fit(X_train, y_train)

# Make predictions
lr_pred = lr_model.predict(X_test)

print("Linear Regression trained successfully!")

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

lr_mae = mean_absolute_error(y_test, lr_pred)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))

lr_r2 = r2_score(y_test, lr_pred)

print("Linear Regression Evaluation")
print("----------------------------")
print("MAE:", lr_mae)
print("RMSE:", lr_rmse)
print("R2 Score:", lr_r2)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Random Forest",
        "Linear Regression"
    ],
    "MAE": [
        mae,
        lr_mae
    ],
    "RMSE": [
        rmse,
        lr_rmse
    ],
    "R2 Score": [
        r2,
        lr_r2
    ]
})

print(comparison)

In [ ]:
comparison.set_index("Model")[["MAE", "RMSE"]].plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Forecasting Model Comparison")
plt.ylabel("Error")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()

In [ ]:
# Export cleaned daily data
daily_df.to_csv("daily_electricity_forecasting.csv")

print("CSV exported successfully!")

In [ ]:
from google.colab import files

files.download("daily_electricity_forecasting.csv")

In [ ]:
# Create dashboard dataset
dashboard_df = daily_df.copy()

# Add prediction columns
dashboard_df["RF_Predicted"] = np.nan
dashboard_df["LR_Predicted"] = np.nan

# Add predictions for test period
dashboard_df.loc[X_test.index, "RF_Predicted"] = y_pred
dashboard_df.loc[X_test.index, "LR_Predicted"] = lr_pred

# Save dashboard dataset
dashboard_df.to_csv("smart_grid_dashboard.csv")

print("Dashboard dataset created successfully!")
print(dashboard_df.head())

In [ ]:
from google.colab import files

files.download("smart_grid_dashboard.csv")

In [ ]:
import pandas as pd

# Load dashboard CSV
dashboard_df = pd.read_csv(
    "smart_grid_dashboard.csv"
)

# Convert Datetime
dashboard_df["Datetime"] = pd.to_datetime(
    dashboard_df["Datetime"]
)

print("Dataset loaded successfully!")
print(dashboard_df.head())
print(dashboard_df.columns)

In [ ]:
print("ELECTRICITY CONSUMPTION DASHBOARD")
print("----------------------------------")

print("Total Records:", len(dashboard_df))

print(
    "Average Power:",
    round(dashboard_df["Power"].mean(), 3),
    "kW"
)

print(
    "Maximum Power:",
    round(dashboard_df["Power"].max(), 3),
    "kW"
)

print(
    "Minimum Power:",
    round(dashboard_df["Power"].min(), 3),
    "kW"
)

In [ ]:
import plotly.express as px
import ipywidgets as widgets

from IPython.display import display, clear_output

# Create month options
months = sorted(
    dashboard_df["Month"].dropna().unique()
)

month_dropdown = widgets.Dropdown(
    options=["All"] + months,
    description="Month:"
)

output = widgets.Output()

def update_dashboard(change=None):

    with output:
        clear_output(wait=True)

        selected_month = month_dropdown.value

        filtered = dashboard_df.copy()

        if selected_month != "All":
            filtered = filtered[
                filtered["Month"] == selected_month
            ]

        print("Records:", len(filtered))

        print(
            "Average Power:",
            round(filtered["Power"].mean(), 3),
            "kW"
        )

        fig = px.line(
            filtered,
            x="Datetime",
            y="Power",
            title="Electricity Consumption Dashboard"
        )

        fig.show()

month_dropdown.observe(
    update_dashboard,
    names="value"
)

display(month_dropdown)
display(output)

update_dashboard()

In [ ]:
print(
    dashboard_df[
        ["Power", "RF_Predicted", "LR_Predicted"]
    ].notna().sum()
)

In [ ]:
prediction_df = dashboard_df.dropna(
    subset=[
        "RF_Predicted",
        "LR_Predicted"
    ]
)

print("Prediction Records:", len(prediction_df))

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

# Actual Power
fig.add_trace(go.Scatter(
    x=prediction_df["Datetime"],
    y=prediction_df["Power"],
    mode="lines",
    name="Actual Power"
))

# Random Forest Prediction
fig.add_trace(go.Scatter(
    x=prediction_df["Datetime"],
    y=prediction_df["RF_Predicted"],
    mode="lines",
    name="Random Forest"
))

# Linear Regression Prediction
fig.add_trace(go.Scatter(
    x=prediction_df["Datetime"],
    y=prediction_df["LR_Predicted"],
    mode="lines",
    name="Linear Regression"
))

fig.update_layout(
    title="Actual vs Predicted Electricity Consumption",
    xaxis_title="Date",
    yaxis_title="Power (kW)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Date range
start_date = widgets.DatePicker(
    description="Start Date"
)

end_date = widgets.DatePicker(
    description="End Date"
)

# Apply button
apply_button = widgets.Button(
    description="Apply Filter",
    button_style="primary"
)

output = widgets.Output()

In [ ]:
def apply_filter(b):

    with output:
        clear_output(wait=True)

        filtered = prediction_df.copy()

        # Start date filter
        if start_date.value:
            filtered = filtered[
                filtered["Datetime"].dt.date >= start_date.value
            ]

        # End date filter
        if end_date.value:
            filtered = filtered[
                filtered["Datetime"].dt.date <= end_date.value
            ]

        if filtered.empty:
            print("No data available.")
            return

        print("Filtered Records:", len(filtered))

        fig = px.line(
            filtered,
            x="Datetime",
            y=["Power", "RF_Predicted", "LR_Predicted"],
            title="Filtered Actual vs Predicted Power"
        )

        fig.show()


apply_button.on_click(apply_filter)

display(
    widgets.VBox([
        start_date,
        end_date,
        apply_button,
        output
    ])
)

In [ ]:
print("Available Start Date:",
      prediction_df["Datetime"].min().date())

print("Available End Date:",
      prediction_df["Datetime"].max().date())

In [ ]:
from IPython.display import display

display(fig)

In [ ]:
        fig = px.line(
            prediction_df,
            x="Datetime",
            y=["Power", "RF_Predicted", "LR_Predicted"],
            title="Filtered Actual vs Predicted Power"
        )

        display(fig)

In [ ]:
import plotly.graph_objects as go
from IPython.display import display

# Calculate KPIs
avg_power = prediction_df["Power"].mean()
max_power = prediction_df["Power"].max()
min_power = prediction_df["Power"].min()
total_records = len(prediction_df)

# Create KPI cards
fig = go.Figure()

kpis = [
    ("Average Power", avg_power),
    ("Maximum Power", max_power),
    ("Minimum Power", min_power),
    ("Prediction Records", total_records)
]

for i, (title, value) in enumerate(kpis):
    fig.add_trace(go.Indicator(
        mode="number",
        value=value,
        title={"text": title},
        domain={
            "row": 0,
            "column": i
        }
    ))

fig.update_layout(
    grid={
        "rows": 1,
        "columns": 4
    },
    title="Electricity Consumption KPI Dashboard",
    height=300
)

fig.show()